In [2]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
from sahi.slicing import slice_coco
from skmultilearn.model_selection import iterative_train_test_split
from tqdm import tqdm

In [3]:
MSGO_CLASSES = {
    "Plane": 0,
    "Bridge": 1,
    "Intersection": 2,
    "Roundabout": 3,
    "Vehicle": 4,
    "Ship": 5,
}

NUM_CLASSES = len(MSGO_CLASSES)
MSGO_CLASSES_REVERSED = {v: k for k, v in MSGO_CLASSES.items()}

In [4]:
def build_image_to_counts(root_dir: str) -> dict[str, dict[int, int]]:
    root_path = Path(root_dir)
    image_to_counts = {}

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    label_files = list(labels_dir.glob("*.txt"))
    for label_file in tqdm(label_files, desc="Counting"):
        img_file = images_dir / f"{label_file.stem}.jpg"

        counts = {}
        with open(label_file) as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            counts[class_id] = counts.get(class_id, 0) + 1

        image_to_counts[str(img_file)] = counts

    return image_to_counts

In [6]:
image_to_counts = build_image_to_counts("D:\\stuff\\datasets\\MSGOv2\\MSGOv2")

Counting: 100%|██████████| 37366/37366 [00:07<00:00, 4916.02it/s]


In [9]:
image_paths = list(image_to_counts.keys())
num_images = len(image_paths)

y_counts = np.zeros((num_images, NUM_CLASSES), dtype=int)
for i, path in enumerate(image_paths):
    counts = image_to_counts[path]
    for class_id, count in counts.items():
        y_counts[i, class_id] = count

X = np.array(image_paths).reshape(-1, 1)

In [10]:
X_train, y_train_counts, X_temp, y_temp_counts = iterative_train_test_split(X, y_counts, test_size=0.2)
X_val, y_val_counts, X_test, y_test_counts = iterative_train_test_split(X_temp, y_temp_counts, test_size=0.5)

X_train_paths = X_train.flatten().tolist()
X_val_paths = X_val.flatten().tolist()
X_test_paths = X_test.flatten().tolist()

print(f"Total images: {len(X)}")
print(f"Train images: {len(X_train_paths)} ({len(X_train_paths) / len(X):.1%})")
print(f"Validation images: {len(X_val_paths)} ({len(X_val_paths) / len(X):.1%})")
print(f"Test images: {len(X_test_paths)} ({len(X_test_paths) / len(X):.1%})")

Total images: 37366
Train images: 29967 (80.2%)
Validation images: 3697 (9.9%)
Test images: 3702 (9.9%)


In [15]:
def check_distribution(paths, image_to_counts_map, num_classes):
    total_counts = np.zeros(num_classes, dtype=int)
    for path in paths:
        counts = image_to_counts_map.get(path, {})
        for class_id, count in counts.items():
            total_counts[class_id] += count
    return total_counts


train_counts = check_distribution(X_train_paths, image_to_counts, NUM_CLASSES)
val_counts = check_distribution(X_val_paths, image_to_counts, NUM_CLASSES)
test_counts = check_distribution(X_test_paths, image_to_counts, NUM_CLASSES)
total_counts = train_counts + val_counts + test_counts

print(f"Class Names: {list(MSGO_CLASSES.keys())}")
print(f"Total Instances: {total_counts}")
print(f"Train Instances: {train_counts} ({(train_counts / total_counts * 100).round(1)}%)")
print(f"Val Instances:   {val_counts} ({(val_counts / total_counts * 100).round(1)}%)")
print(f"Test Instances:  {test_counts} ({(test_counts / total_counts * 100).round(1)}%)")

Class Names: ['Plane', 'Bridge', 'Intersection', 'Roundabout', 'Vehicle', 'Ship']
Total Instances: [ 80264  10027  10156   1718 711526 113774]
Train Instances: [ 59140   7954   8137   1277 533667  80490] ([73.7 79.3 80.1 74.3 75.  70.7]%)
Val Instances:   [10636   997   971   196 94430 18969] ([13.3  9.9  9.6 11.4 13.3 16.7]%)
Test Instances:  [10488  1076  1048   245 83429 14315] ([13.1 10.7 10.3 14.3 11.7 12.6]%)


In [ ]:
def yolo_hbb_to_coco(
    yolo_coords_norm: list[float], img_width: int, img_height: int
) -> tuple[list[float], list[float], float]:
    x_center_norm, y_center_norm, width_norm, height_norm = yolo_coords_norm

    width_abs = width_norm * img_width
    height_abs = height_norm * img_height
    x_center_abs = x_center_norm * img_width
    y_center_abs = y_center_norm * img_height

    x_min = x_center_abs - (width_abs / 2)
    y_min = y_center_abs - (height_abs / 2)

    coco_bbox = [
        round(x_min, 2),
        round(y_min, 2),
        round(width_abs, 2),
        round(height_abs, 2),
    ]

    x_max, y_max = x_min + width_abs, y_min + height_abs
    coco_segmentation = [
        round(x_min, 2),
        round(y_min, 2),
        round(x_max, 2),
        round(y_min, 2),
        round(x_max, 2),
        round(y_max, 2),
        round(x_min, 2),
        round(y_max, 2),
    ]

    area = width_abs * height_abs

    return coco_segmentation, coco_bbox, round(area, 2)


def create_master_coco_json_from_hbb(root_dir: str, show_bad_annotations: bool = False) -> None:
    coco_data = {
        "info": {"description": "Pre-sliced MSGO HBB dataset"},
        "licenses": [],
        "categories": [
            {"id": cid, "name": cname, "supercategory": "object"} for cid, cname in MSGO_CLASSES_REVERSED.items()
        ],
        "images": [],
        "annotations": [],
    }

    root_path = Path(root_dir)
    image_id_counter, annotation_id_counter = 1, 1
    skipped_annotations_count = 0

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    label_files = list(labels_dir.glob("*.txt"))

    for label_file in tqdm(label_files, desc="Processing"):
        img_file = images_dir / f"{label_file.stem}.jpg"

        with Image.open(img_file) as img:
            img_width, img_height = img.size

        image_info = {
            "id": image_id_counter,
            "file_name": img_file.relative_to(root_path).as_posix(),
            "width": img_width,
            "height": img_height,
        }
        coco_data["images"].append(image_info)

        with open(label_file) as f:
            lines = f.readlines()

        for line_num, line in enumerate(lines, 1):
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            yolo_obb_data = [float(p) for p in parts[1:]]

            segmentation, bbox, area = yolo_hbb_to_coco(yolo_obb_data, img_width, img_height)

            reason = ""
            if bbox[2] <= 0 or bbox[3] <= 0 or area <= 1:
                skipped_annotations_count += 1
                if show_bad_annotations:
                    print(f"\nFile: {label_file.name}")
                    print(f"Line in File: {line_num}")
                    print(f"Reason: {reason}")
                    print(f"Bbox [x,y,w,h]: {bbox}")
                    print(f"Area: {area}")
                    print(f"Segmentation: {segmentation}")
                continue

            annotation_info = {
                "id": annotation_id_counter,
                "image_id": image_id_counter,
                "category_id": class_id,
                "bbox": bbox,
                "segmentation": [segmentation],
                "area": area,
                "iscrowd": 0,
            }
            coco_data["annotations"].append(annotation_info)
            annotation_id_counter += 1

        image_id_counter += 1

    print(f"\nProcessed {image_id_counter - 1} images and {annotation_id_counter - 1} annotations.")
    print(f"Skipped {skipped_annotations_count} annotations.")

    ouput_json_path = root_path / "master_annotations.coco.json"
    with open(ouput_json_path, "w") as f:
        json.dump(coco_data, f)


create_master_coco_json_from_hbb("D:\\stuff\\datasets\\MSGOv2\\MSGOv2", False)

Processing: 100%|██████████| 37366/37366 [00:34<00:00, 1072.31it/s]



Processed 37366 images and 1009338 annotations.
Skipped 183 annotations.


In [ ]:
ROOT_DIR = Path("D:/stuff/datasets/MSGOv2/MSGOv2")
MASTER_COCO_PATH = ROOT_DIR / "master_annotations.coco.json"
FINAL_DATASET_DIR = ROOT_DIR / "sliced"


def slice_split(image_paths, master_coco_data, split_name):
    output_dir = FINAL_DATASET_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {split_name} split...")

    split_relative_paths = {Path(p).relative_to(ROOT_DIR).as_posix() for p in image_paths}
    split_images = [img for img in master_coco_data["images"] if img["file_name"] in split_relative_paths]
    split_image_ids = {img["id"] for img in split_images}
    split_annotations = [ann for ann in master_coco_data["annotations"] if ann["image_id"] in split_image_ids]

    subset_coco_data = {
        "images": split_images,
        "annotations": split_annotations,
        "categories": master_coco_data["categories"],
    }

    subset_coco_path = output_dir / f"{split_name}_subset.json"
    with open(subset_coco_path, "w") as f:
        json.dump(subset_coco_data, f)

    slice_coco(
        coco_annotation_file_path=subset_coco_path,
        image_dir=ROOT_DIR,
        output_dir=output_dir,
        output_coco_annotation_file_name="_annotations",
        slice_height=800,
        slice_width=800,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        min_area_ratio=0.4,
        ignore_negative_samples=False,
        verbose=False,
    )

    subset_coco_path.unlink()

In [16]:
with open(MASTER_COCO_PATH) as f:
    master_data = json.load(f)

slice_split(X_test_paths, master_data, "test")
slice_split(X_val_paths, master_data, "val")
slice_split(X_train_paths, master_data, "train")


Processing test split...


100%|██████████| 3702/3702 [07:50<00:00,  7.87it/s]  



Processing val split...


100%|██████████| 3697/3697 [13:15<00:00,  4.65it/s]   



Processing train split...


100%|██████████| 29967/29967 [1:24:16<00:00,  5.93it/s]  


In [21]:
def empty_image_percentages(root_dir):
    root_path = Path(root_dir)
    results = {}
    total_empty = 0
    total_images = 0

    for split_path in root_path.iterdir():
        images_dir = split_path / "images"
        labels_dir = split_path / "labels"

        split_total = 0
        split_empty = 0

        for img_file in images_dir.iterdir():
            split_total += 1
            img_name = img_file.stem
            label_file = labels_dir / f"{img_name}.txt"
            if label_file.exists() and label_file.stat().st_size == 0:
                split_empty += 1

        results[split_path.name] = {
            "empty_count": split_empty,
            "total": split_total,
            "percentage": round((split_empty / split_total) * 100, 2),
        }

        total_empty += split_empty
        total_images += split_total

    overall_percentage = round((total_empty / total_images) * 100, 2)

    for split, stats in results.items():
        print(f"{split}: {stats['empty_count']} / {stats['total']} ({stats['percentage']}%)")

    print(f"\nOverall: {total_empty} / {total_images} ({overall_percentage}%)")

    return results, overall_percentage


empty_image_percentages("D:\\stuff\\datasets\\MSGOv2\\MSGOv2\\sliced")

test: 391 / 7637 (5.12%)
train: 4059 / 79938 (5.08%)
val: 397 / 7670 (5.18%)

Overall: 4847 / 95245 (5.09%)


({'test': {'empty_count': 391, 'total': 7637, 'percentage': 5.12},
  'train': {'empty_count': 4059, 'total': 79938, 'percentage': 5.08},
  'val': {'empty_count': 397, 'total': 7670, 'percentage': 5.18}},
 5.09)

In [ ]:
MSGO_CLASSES = {
    "Plane": 0,
    "Bridge": 1,
    "Intersection": 2,
    "Roundabout": 3,
    "Vehicle": 4,
    "Ship": 5,
}
NUM_CLASSES = len(MSGO_CLASSES)
CLASS_NAMES = list(MSGO_CLASSES.keys())


def verify_final_distribution(root_dir: str):
    root_path = Path(root_dir)
    all_split_data = {}
    splits_to_process = [d for d in root_path.iterdir() if d.is_dir()]

    for split_path in splits_to_process:
        split_name = split_path.name
        labels_dir = split_path / "labels"

        split_counts = np.zeros(NUM_CLASSES, dtype=int)
        annotated_file_count = 0
        empty_file_count = 0

        label_files = list(labels_dir.glob("*.txt"))

        for label_file in tqdm(label_files, desc=f"Analyzing '{split_name}' labels"):
            with open(label_file) as f:
                lines = f.readlines()

            if not lines:
                empty_file_count += 1
            else:
                annotated_file_count += 1
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        if 0 <= class_id < NUM_CLASSES:
                            split_counts[class_id] += 1

        all_split_data[split_name] = {
            "counts": split_counts,
            "annotated_files": annotated_file_count,
            "empty_files": empty_file_count,
        }

    total_counts = np.zeros(NUM_CLASSES, dtype=int)
    for data in all_split_data.values():
        total_counts += data["counts"]

    print(f"Class Names: {CLASS_NAMES}")
    print(f"Total Instances: {total_counts}")

    for split_name, data in all_split_data.items():
        split_counts = data["counts"]
        percentages = np.round((split_counts / (total_counts + 1e-9)) * 100, 1)
        print(f"{split_name.capitalize():<6} Instances: {split_counts} ({percentages}%)")

    for split_name, data in all_split_data.items():
        annotated = data["annotated_files"]
        empty = data["empty_files"]
        total = annotated + empty
        print(
            f"{split_name.capitalize():<6}: {annotated} images with annotations, {empty} empty images (Total: {total})"
        )


verify_final_distribution("D:\\stuff\\datasets\\MSGOv2\\sliced")